In [ ]:
# Install Required Libraries

!pip install -q anthropic

In [ ]:
# Import Libraries

import os, re, time
import pandas as pd
import anthropic

from tqdm import tqdm

In [ ]:
# Set Claude API Key

client = anthropic.Anthropic(api_key = "")

In [ ]:
# List All Currently Available Models via Claude API

available_models = client.models.list()

for model in available_models.data:
    print(model.id)

In [ ]:
# Define Prompt Template

def build_prompt(context, sentence_A, sentence_B, sentence_C):
    return f"""\
Using the 7-point scale below, rate the acceptability of the following sentence given the context:

1. Strongly Unacceptable
2. Unacceptable
3. Somewhat Unacceptable
4. Neutral
5. Somewhat Acceptable
6. Acceptable
7. Strongly Acceptable

Context:
{context.strip()}

Sentence A: "{sentence_A}"
Sentence B: "{sentence_B}"
Sentence C: "{sentence_C}"

Just provide a numerical rating (1–7) for Sentence A, Sentence B, and Sentence C, respectively.
"""

In [ ]:
# Function to Extract Ratings

def extract_ratings(text):
    # Try to Match Labeled Responses First
    match_a = re.search(r"A:\s*([1-7])", text)
    match_b = re.search(r"B:\s*([1-7])", text)
    match_c = re.search(r"C:\s*([1-7])", text)

    if match_a and match_b and match_c:
        return match_a.group(1), match_b.group(1), match_c.group(1)

    # Fallback: Match First Three Standalone 1–7 Digits
    numbers = re.findall(r"\b[1-7]\b", text)
    if len(numbers) >= 3:
        return numbers[0], numbers[1], numbers[2]

    # Final Fallback if Nothing Useful Found
    return "ERROR", "ERROR", "ERROR"

In [ ]:
# Models

models = [
    "claude-opus-4-8",
    "claude-opus-4-7",
    "claude-opus-4-6",
    "claude-sonnet-4-6",
    "claude-haiku-4-5-20251001"
]

In [ ]:
# Test on Individual Sentences

sample_df = pd.read_csv("./exp2_event.csv")

sample_context = sample_df.loc[0, "CONTEXT"]
sample_A = sample_df.loc[0, "A"]
sample_B = sample_df.loc[0, "B"]
sample_C = sample_df.loc[0, "C"]

prompt = build_prompt(sample_context, sample_A, sample_B, sample_C)

response = client.messages.create(
    model = "", # Model ID
    max_tokens = 1500,
    messages = [{"role": "user", "content": prompt}]
)

output = response.content[0].text.strip()
print(output)
print(f"stop_reason: {response.stop_reason}")
print(f"input_tokens: {response.usage.input_tokens}")
print(f"output_tokens: {response.usage.output_tokens}")

In [ ]:
# Run Experiment for Each Context File

context_files = {
    "object": "./exp2_object.csv",
    "goal":   "./exp2_goal.csv",
    "event":  "./exp2_event.csv"
}

for context_type, filepath in context_files.items():
    print(f"\n=== Context: {context_type} ===")

    df = pd.read_csv(filepath)
    combined_results = df.copy()

    for model in models:
        combined_results[f"Rating_A_{model}"] = ""
        combined_results[f"Rating_B_{model}"] = ""
        combined_results[f"Rating_C_{model}"] = ""

    for model_name in models:
        print(f"Running model: {model_name}")

        for i in tqdm(range(len(df)), desc = f"Processing ({model_name})"):
            prompt = build_prompt(
                df.loc[i, "CONTEXT"],
                df.loc[i, "A"],
                df.loc[i, "B"],
                df.loc[i, "C"]
            )

            try:
                response = client.messages.create(
                    model = model_name,
                    max_tokens = 1500,
                    messages = [{"role": "user", "content": prompt}]
                )
                output = response.content[0].text.strip()
                rating_a, rating_b, rating_c = extract_ratings(output)
            except Exception as e:
                rating_a, rating_b, rating_c = "ERROR", "ERROR", "ERROR"
                print(f"Error at idx {i}: {e}")

            combined_results.at[i, f"Rating_A_{model_name}"] = rating_a
            combined_results.at[i, f"Rating_B_{model_name}"] = rating_b
            combined_results.at[i, f"Rating_C_{model_name}"] = rating_c

            time.sleep(1.5)

            # Save Intermediate Results Every 50 Rows
            if i % 50 == 0:
                combined_results.to_csv(f"./exp2_claude_{context_type}_temp.csv", index = False)

    # Save Final Results for Each Context
    combined_results.to_csv(f"./exp2_claude_{context_type}.csv", index = False)
    print(f"Saved: exp2_claude_{context_type}.csv")